# ACE-iPred — complete training and analysis notebook

This notebook consolidates the **ACE-iPred training, frozen-model reproduction, cross-validation, independent-test evaluation, paired comparator analysis, and SHAP interpretation** available in the supplied reproducibility package.

The generated motif-excluded sequences are treated here as **negative examples assigned to the negative class for model training**.

> **Scope note:** the package contains the frozen 10-feature panel and model hyperparameters, but it does not contain an ACE-iPred-specific upstream feature-screening trajectory workbook analogous to the ScavenPred screening workbooks. Therefore, this notebook reproduces all ACE-iPred analyses that are supported by the supplied ACE-iPred files and does not invent an unavailable feature-selection history.


In [ ]:
from pathlib import Path
import warnings, json, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, auc
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

# Notebook is stored under ACE-iPred/notebooks/
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"

print("ACE-iPred root:", ROOT.resolve())
print("Data files:", len(list(DATA.glob("*.xlsx"))))
print("Result files:", len(list(RESULTS.glob("*.xlsx"))))


## 1. Frozen model specification

In [ ]:
FEATURES = ['BLOSUM4', 'BLOSUM1', 'KF5', 'VHSE2', 'BLOSUM10',
            'SVGER2', 'SVGER3', 'ProtFP7', 'SVGER4', 'MSWHIM3']

MODEL_PARAMS = {
    'n_estimators': 200,
    'max_depth': 10,
    'min_samples_leaf': 4,
    'max_features': 1.0,
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs': -1
}

print("Selected frozen features (n=%d):" % len(FEATURES))
print(FEATURES)
print("\nFrozen Random Forest parameters:")
print(json.dumps(MODEL_PARAMS, indent=2))


## 2. Dataset curation audit supplied with the package

In [ ]:
audit_path = DATA / "ACE_iPred_similarity_cleaned.xlsx"
summary = pd.read_excel(audit_path, sheet_name="summary")
overlap = pd.read_excel(audit_path, sheet_name="exact_overlap_allocation")
conflicts = pd.read_excel(audit_path, sheet_name="conflicts_removed")
sim_removed = pd.read_excel(audit_path, sheet_name="similarity_removed")

print("Curation summary:")
display(summary)
print("\nExact-overlap allocations:", len(overlap))
display(overlap.head(10))
print("\nConflicting sequences removed:", len(conflicts))
display(conflicts.head(10))
print("\nSimilarity-based test removals:", len(sim_removed))
display(sim_removed)


## 3. Load the cleaned training set and 88-descriptor feature matrix

In [ ]:
feat = pd.read_excel(DATA / "ACE_feat.xlsx")
labels = pd.read_excel(audit_path, sheet_name="train_cleaned")

feat["_seq"] = feat["seq"].astype(str).str.strip().str.upper()
labels["_seq"] = labels["seq"].astype(str).str.strip().str.upper()
label_map = dict(zip(labels["_seq"], labels["class"]))

y = feat["_seq"].map(label_map)
assert y.notna().all(), "Some feature rows do not have a cleaned training label."
y = y.astype(int)
X = feat[FEATURES].copy()

print("Training feature matrix:", feat.shape)
print("Frozen model matrix X:", X.shape)
print("Training labels:", y.shape)
print("Class counts:", y.value_counts().sort_index().to_dict())
print("Missing values in selected features:", int(X.isna().sum().sum()))
display(X.describe().T)


## 4. Five-fold stratified cross-validation on the cleaned training set

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "mcc": __import__("sklearn").metrics.make_scorer(matthews_corrcoef)
}
cv_model = RandomForestClassifier(**MODEL_PARAMS)
cvres = cross_validate(cv_model, X, y, cv=cv, scoring=scoring, n_jobs=-1)

rows = []
for metric in scoring:
    vals = cvres["test_" + metric]
    rows.append([metric, vals.mean(), vals.std(ddof=1), *vals])
cv_table = pd.DataFrame(rows, columns=["Metric","Mean","SD","Fold1","Fold2","Fold3","Fold4","Fold5"])
display(cv_table)


In [ ]:
cv_pred = cross_val_predict(RandomForestClassifier(**MODEL_PARAMS), X, y, cv=cv, method="predict", n_jobs=-1)
cv_prob = cross_val_predict(RandomForestClassifier(**MODEL_PARAMS), X, y, cv=cv, method="predict_proba", n_jobs=-1)[:,1]
cm_cv = confusion_matrix(y, cv_pred)
print("Cross-validated confusion matrix [[TN, FP], [FN, TP]]:")
print(cm_cv)
print("\nCross-validated classification report:")
print(classification_report(y, cv_pred, digits=4))
print("MCC:", round(matthews_corrcoef(y, cv_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y, cv_prob), 4))


## 5. Fit the frozen Random Forest on all cleaned training examples

In [ ]:
rf = RandomForestClassifier(**MODEL_PARAMS)
rf.fit(X, y)
model_path = RESULTS / "frozen_random_forest_complete_notebook.joblib"
joblib.dump(rf, model_path)
print("Model fitted on", len(X), "training examples.")
print("Saved:", model_path)


## 6. Independent test set after UniDL4BioPep overlap removal (n = 253)

In [ ]:
test = pd.read_excel(DATA / "ACE_Test_feat_UniDL4BioPep_overlap_removed.xlsx", sheet_name="Sheet1")
X_test = test[FEATURES].copy()
y_test = test["class"].astype(int)

prob = rf.predict_proba(X_test)[:,1]
pred = (prob >= 0.5).astype(int)

metrics = {
    "Accuracy": accuracy_score(y_test, pred),
    "Balanced accuracy": balanced_accuracy_score(y_test, pred),
    "Precision": precision_score(y_test, pred),
    "Recall/Sensitivity": recall_score(y_test, pred),
    "Specificity": confusion_matrix(y_test, pred).ravel()[0] / confusion_matrix(y_test, pred).ravel()[:2].sum(),
    "F1": f1_score(y_test, pred),
    "MCC": matthews_corrcoef(y_test, pred),
    "ROC-AUC": roc_auc_score(y_test, prob),
}
print("Independent test size:", len(test))
print("Class counts:", y_test.value_counts().sort_index().to_dict())
display(pd.DataFrame(metrics.items(), columns=["Metric","Value"]))
print("\nConfusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, pred))
print("\nClassification report:")
print(classification_report(y_test, pred, digits=4))


## 7. Compare reproduced independent-test results with the packaged reevaluation

In [ ]:
pack_perf = pd.read_excel(RESULTS / "ACE_iPred_253_Reevaluation.xlsx", sheet_name="performance")
pack_counts = pd.read_excel(RESULTS / "ACE_iPred_253_Reevaluation.xlsx", sheet_name="counts")
print("Packaged performance table:")
display(pack_perf)
print("\nPackaged count audit:")
display(pack_counts)


## 8. ROC curve for the reproduced frozen model

In [ ]:
fpr, tpr, _ = roc_curve(y_test, prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"ACE-iPred (AUC={roc_auc:.3f})")
plt.plot([0,1],[0,1], linestyle="--")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ACE-iPred independent-test ROC curve")
plt.legend()
plt.tight_layout()
plt.show()


## 9. Paired comparison with the packaged comparator predictions

In [ ]:
paired_book = RESULTS / "ACE_iPred_Paired_Comparison_253.xlsx"
paired_stats = pd.read_excel(paired_book, sheet_name="paired_statistics")
mcnemar = pd.read_excel(paired_book, sheet_name="mcnemar")
paired_preds = pd.read_excel(paired_book, sheet_name="paired_predictions")

print("Packaged paired statistical comparisons:")
display(paired_stats)
print("\nMcNemar analysis:")
display(mcnemar)
print("\nFirst paired predictions:")
display(paired_preds.head(10))


## 10. SHAP analysis of the fitted frozen model

In [ ]:
import shap

explainer = shap.TreeExplainer(rf)
sv = explainer.shap_values(X)
arr = np.asarray(sv)

if isinstance(sv, list):
    sv_pos = np.asarray(sv[1])
elif arr.ndim == 3:
    sv_pos = arr[:, :, 1]
else:
    sv_pos = arr

importance = pd.DataFrame({
    "Feature": FEATURES,
    "Mean_abs_SHAP": np.abs(sv_pos).mean(axis=0)
}).sort_values("Mean_abs_SHAP", ascending=False)

display(importance)


In [ ]:
plt.figure(figsize=(7,5))
plt.barh(importance["Feature"][::-1], importance["Mean_abs_SHAP"][::-1])
plt.xlabel("Mean |SHAP value|")
plt.title("ACE-iPred global SHAP feature importance")
plt.tight_layout()
plt.show()


In [ ]:
# SHAP beeswarm / summary plot
shap.summary_plot(sv_pos, X, feature_names=FEATURES, show=False)
plt.tight_layout()
plt.show()


## 11. Compare reproduced SHAP ranking with the packaged SHAP workbook

In [ ]:
pack_shap = pd.read_excel(RESULTS / "ACE_iPred_SHAP_Analysis.xlsx", sheet_name="global_importance")
comparison = importance.merge(pack_shap, on="Feature", how="outer", suffixes=("_reproduced","_packaged"))
display(comparison)


## 12. Export the notebook-generated tables

In [ ]:
export_path = RESULTS / "ACE_iPred_complete_notebook_outputs.xlsx"
with pd.ExcelWriter(export_path, engine="openpyxl") as writer:
    cv_table.to_excel(writer, sheet_name="5fold_CV", index=False)
    pd.DataFrame(metrics.items(), columns=["Metric","Value"]).to_excel(writer, sheet_name="independent_test", index=False)
    importance.to_excel(writer, sheet_name="SHAP_importance", index=False)
    pd.DataFrame({
        "seq": test["seq"],
        "class": y_test,
        "probability": prob,
        "prediction": pred
    }).to_excel(writer, sheet_name="test_predictions", index=False)

print("Exported:", export_path)


## 13. Reproducibility summary

This notebook shows, with executable code and visible outputs:

1. the frozen ACE-iPred feature panel and Random Forest parameters;
2. the supplied dataset-cleaning audit;
3. cleaned training-set composition;
4. 5-fold stratified cross-validation;
5. full frozen-model training;
6. evaluation on the 253-sequence independent test set;
7. comparison with the packaged reevaluation results;
8. ROC analysis;
9. paired comparison with the packaged comparator;
10. SHAP global and local-distribution analyses; and
11. export of all notebook-generated numerical outputs.

The negative examples are handled strictly as **sequences assigned to the negative class for model training**.
